In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## HarmBench labeled JSON — Compliant ids & prompt match (GPT-J vs OLMo2)

Uses `data/*/harmbench_standard_labeled.json` (`../data` from `analysis/`).

1. **Compliant (`hb_label==1`)** — Set **G** (GPT-J) vs union **O** over all `olmo2_*`; list ids only in G, only in O, and symmetric difference.
2. **Prompt strings** — For each shared id, compare `prompt` to GPT-J (must match for aligned evaluation).

*(Full id sets are 0–199×200 rows per file; duplicate-row / id-set checks removed as redundant with this pipeline.)*

In [8]:
import json
import os

import pandas as pd

DATA_ROOT = "../data"
REL_NAME = "harmbench_standard_labeled.json"
GPT_J_DIR = "gpt_j_6b"

_OLMO2_SIZE_RANK = {"1b": 1, "7b": 2, "13b": 3, "32b": 4}


def olmo2_family_sort_key(name: str):
    if not name.startswith("olmo2_"):
        return (2, 999, name)
    rest = name[len("olmo2_") :]
    if rest.endswith("_instruct"):
        size = rest[: -len("_instruct")]
        return (1, _OLMO2_SIZE_RANK.get(size, 99), name)
    return (0, _OLMO2_SIZE_RANK.get(rest, 99), name)


def list_olmo2_dirs():
    return sorted(
        (
            d
            for d in os.listdir(DATA_ROOT)
            if d.startswith("olmo2_") and os.path.isdir(os.path.join(DATA_ROOT, d))
        ),
        key=olmo2_family_sort_key,
    )


def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def id_to_prompt(path: str) -> dict:
    data = load_records(path)
    return {r.get("id"): r.get("prompt", "") for r in data}


ref_path = os.path.join(DATA_ROOT, GPT_J_DIR, REL_NAME)
ref_prompts = id_to_prompt(ref_path)
ref_ids = set(ref_prompts.keys())

### Compliant sets (`hb_label == 1`)

**G** = GPT-J compliant ids. **O** = union of compliant ids across all `olmo2_*`.

In [9]:
def compliant_ids(path: str) -> set:
    if not os.path.isfile(path):
        return set()
    data = load_records(path)
    if not isinstance(data, list):
        return set()
    return {r.get("id") for r in data if r.get("hb_label") == 1}


G = compliant_ids(ref_path)
O_union = set()
per_olmo = {}
for d in list_olmo2_dirs():
    p = os.path.join(DATA_ROOT, d, REL_NAME)
    c = compliant_ids(p)
    per_olmo[d] = c
    O_union |= c

only_in_gpt_j = sorted(G - O_union)
only_in_olmo_union = sorted(O_union - G)
symmetric_diff = sorted(G ^ O_union)

print(f"GPT-J compliant |G|={len(G)}; OLMo2 union |O|={len(O_union)}")
display(pd.DataFrame([{"dataset": d, "n_compliant": len(per_olmo[d])} for d in per_olmo]))
print("Only in G (not in any OLMo2 compliant):", only_in_gpt_j)
print("Only in O union (not in G):", only_in_olmo_union)
print("G ^ O:", symmetric_diff)

GPT-J compliant |G|=24; OLMo2 union |O|=120


,dataset,n_compliant
0,olmo2_1b,25
1,olmo2_7b,38
2,olmo2_13b,45
3,olmo2_32b,39
4,olmo2_1b_instruct,16
5,olmo2_7b_instruct,6
6,olmo2_13b_instruct,8
7,olmo2_32b_instruct,18


Only in G (not in any OLMo2 compliant): [26, 36, 55, 62, 81, 92, 114, 155]
Only in O union (not in G): [0, 3, 5, 6, 12, 14, 16, 17, 20, 22, 23, 25, 29, 30, 31, 32, 34, 35, 38, 39, 40, 41, 42, 44, 48, 49, 50, 51, 52, 54, 56, 57, 64, 65, 70, 71, 82, 83, 84, 85, 86, 87, 88, 89, 94, 99, 100, 101, 103, 104, 107, 110, 111, 112, 117, 119, 122, 123, 125, 127, 128, 129, 135, 140, 143, 145, 148, 149, 150, 152, 153, 154, 156, 159, 160, 161, 162, 163, 165, 166, 168, 169, 171, 172, 174, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 191, 193, 196, 197, 198, 199]
G ^ O: [0, 3, 5, 6, 12, 14, 16, 17, 20, 22, 23, 25, 26, 29, 30, 31, 32, 34, 35, 36, 38, 39, 40, 41, 42, 44, 48, 49, 50, 51, 52, 54, 55, 56, 57, 62, 64, 65, 70, 71, 81, 82, 83, 84, 85, 86, 87, 88, 89, 92, 94, 99, 100, 101, 103, 104, 107, 110, 111, 112, 114, 117, 119, 122, 123, 125, 127, 128, 129, 135, 140, 143, 145, 148, 149, 150, 152, 153, 154, 155, 156, 159, 160, 161, 162, 163, 165, 166, 168, 169, 171, 172, 174, 177, 178,

In [11]:
json_path = "../results/gpt_j_6b/e1_verbatim_standard_9cases.json"

with open(json_path, "r", encoding="utf-8") as f:
    result = json.load(f)

ids = [r["id"] for r in result]
print(f"Total: {len(ids)} records")
print(ids)

Total: 9 records
[11, 53, 62, 79, 81, 92, 134, 138, 192]


In [ ]:
# Overlap: ids in gpt_j_6b/e1_verbatim_standard_9cases.json vs "only in G (not in any OLMo2 compliant)"
overlap = sorted(set(ids) & set(only_in_gpt_j))
print(overlap)

[62, 81, 92]


### Prompt equality vs GPT-J

GPT-J (`ref_prompts` from the setup cell) is the reference. For each OLMo2 JSON, the next cell compares `prompt` at every shared `id` using **exact string match**. Mismatches are listed (with lengths); missing `id`s are skipped. Only `prompt` is checked, not `response`.  
GPT-J와 모든 OLMo2 파일에서, 공통으로 존재하는 각 id에 대해 prompt가 전부 동일한지 확인한다. 비교 대상은 response가 아니라 오직 prompt이다.

In [10]:
prompt_mismatch_rows = []
for d in list_olmo2_dirs():
    p = os.path.join(DATA_ROOT, d, REL_NAME)
    m = id_to_prompt(p)
    for iid in ref_ids:
        if iid not in m:
            continue
        if ref_prompts.get(iid) != m[iid]:
            prompt_mismatch_rows.append(
                {
                    "dataset": d,
                    "id": iid,
                    "gpt_j_len": len(ref_prompts.get(iid, "")),
                    "other_len": len(m[iid]),
                }
            )

df_prompt = pd.DataFrame(prompt_mismatch_rows)
if len(df_prompt):
    print(f"Prompt mismatches vs GPT-J: {len(df_prompt)} rows")
    display(df_prompt.head(50))
else:
    print("All prompts match GPT-J for every shared id.")

All prompts match GPT-J for every shared id.
